# 04 - Queue Delay Analysis

Analyze queue delay distribution and correlation with system load.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

try:
    %store -r df
    print(f"Loaded {len(df):,} events")
except:
    np.random.seed(42)
    n = 50000
    df = pd.DataFrame({
        "timestamp_utc_iso": pd.date_range("2025-01-01", periods=n, freq="10ms"),
        "latency_us": np.random.lognormal(6, 0.5, n).astype(int),
        "queue_delay_us": np.random.exponential(200, n).astype(int),
        "worker_id": np.random.randint(0, 4, n),
    })

if "timestamp" not in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp_utc_iso"])


In [ ]:
# Queue delay distribution
if "queue_delay_us" in df.columns:
    qd = df["queue_delay_us"].dropna()
    
    print("Queue Delay (μs):")
    print(f"  Mean: {qd.mean():.1f}")
    print(f"  Median: {qd.median():.1f}")
    print(f"  p99: {qd.quantile(0.99):.1f}")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(qd, bins=100, edgecolor="white", alpha=0.7)
    axes[0].set_xlabel("Queue Delay (μs)")
    axes[0].set_title("Queue Delay Distribution")
    
    sorted_qd = np.sort(qd)
    cdf = np.arange(1, len(sorted_qd) + 1) / len(sorted_qd)
    axes[1].plot(sorted_qd, cdf, linewidth=2)
    axes[1].set_xlabel("Queue Delay (μs)")
    axes[1].set_title("Queue Delay CDF")
    
    plt.tight_layout()
    plt.show()


# 04 - Queue Delay Analysis

Analyze queue delay distributions and correlations with load.

## Objectives
- Compute queue delay distributions
- Analyze correlation with load
- Identify backpressure patterns
- Compare queue delay across workers


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

EXPERIMENT_ID = "exp_2025_0101_001"
DATA_PATH = f"../data/{EXPERIMENT_ID}/merged/merged.parquet"

df = pd.read_parquet(DATA_PATH)
has_queue_data = "queue_delay_us" in df.columns and df["queue_delay_us"].sum() > 0
print(f"Queue delay data available: {has_queue_data}")


In [ ]:
if has_queue_data:
    delay = df["queue_delay_us"]
    print("Queue Delay Statistics (μs):")
    print(f"  Mean:   {delay.mean():.2f}")
    print(f"  Median: {delay.median():.2f}")
    print(f"  p90:    {delay.quantile(0.90):.2f}")
    print(f"  p99:    {delay.quantile(0.99):.2f}")
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(delay, bins=100, edgecolor="white", alpha=0.7, color="#FF9800")
    ax.axvline(delay.median(), color="red", linestyle="--", label=f"Median: {delay.median():.0f}μs")
    ax.set_xlabel("Queue Delay (μs)")
    ax.set_ylabel("Frequency")
    ax.set_title("Queue Delay Distribution")
    ax.legend()
    plt.show()
else:
    print("No queue delay data in this experiment.")
